<a href="https://colab.research.google.com/github/18217265596/sx/blob/master/LigandMPNN_Colab_Complete_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LigandMPNN — Colab 生产版

本版本仅包含正式生产流程：下载官方当前启用的全部权重、修复新版 Colab 兼容问题、上传 PDB、按编号选择模型权重、运行设计、调用 `extract.py` 提取高分唯一序列并打包结果。

In [ ]:
# 0. 检查运行时
import sys
import platform
import torch

print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
else:
    print("建议在“运行时 → 更改运行时类型”中选择 T4 GPU。")

In [ ]:
# 1. 克隆官方仓库并安装 Colab 兼容依赖
from pathlib import Path
import os
import re
import shutil
import subprocess
import sys

ROOT = Path("/content/LigandMPNN")
RESET_REPOSITORY = True

if RESET_REPOSITORY and ROOT.exists():
    shutil.rmtree(ROOT)

subprocess.run(
    [
        "git", "clone", "--depth", "1",
        "https://github.com/dauparas/LigandMPNN.git",
        str(ROOT),
    ],
    check=True,
)

# 保留 Colab 自带 PyTorch/CUDA，不安装官方 requirements.txt 中锁定的旧环境。
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "ProDy==2.6.1",
        "biopython>=1.81",
        "ml-collections==0.1.1",
        "dm-tree==0.1.8",
    ],
    check=True,
)

print("Repository:", ROOT)
subprocess.run(
    ["git", "-C", str(ROOT), "log", "-1", "--oneline"],
    check=True,
)

In [ ]:
# 2. 下载官方 get_model_params.sh 当前启用的全部 15 个权重
MODEL_DIR = ROOT / "model_params"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run(
    ["bash", str(ROOT / "get_model_params.sh"), str(MODEL_DIR)],
    check=True,
)

weight_files = sorted(MODEL_DIR.glob("*.pt"))
if len(weight_files) != 15:
    raise RuntimeError(
        f"预期下载 15 个权重，实际得到 {len(weight_files)} 个。"
    )

print("Downloaded checkpoints:")
for path in weight_files:
    if path.stat().st_size < 1024**2:
        raise RuntimeError(f"权重文件疑似不完整：{path}")
    print(f"  {path.name}: {path.stat().st_size / 1024**2:.1f} MiB")

In [ ]:
# 3. 修复新版 PyTorch 和 NumPy/OpenFold 兼容性，并获取 extract.py
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

run_py = ROOT / "run.py"
source = run_py.read_text(encoding="utf-8")
source = source.replace(
    "torch.load(checkpoint_path, map_location=device)",
    "torch.load(checkpoint_path, map_location=device, weights_only=False)",
)
source = source.replace(
    "torch.load(args.checkpoint_path_sc, map_location=device)",
    "torch.load(args.checkpoint_path_sc, map_location=device, weights_only=False)",
)
run_py.write_text(source, encoding="utf-8")

numpy_aliases = {
    r"\bnp\.int\b": "int",
    r"\bnp\.float\b": "float",
    r"\bnp\.bool\b": "bool",
    r"\bnp\.object\b": "object",
    r"\bnp\.str\b": "str",
    r"\bnp\.complex\b": "complex",
}

changed = []
for py_file in ROOT.rglob("*.py"):
    text = py_file.read_text(encoding="utf-8")
    patched = text
    for pattern, replacement in numpy_aliases.items():
        patched = re.sub(pattern, replacement, patched)
    if patched != text:
        py_file.write_text(patched, encoding="utf-8")
        changed.append(py_file.relative_to(ROOT))

EXTRACT_PY = ROOT / "extract.py"
subprocess.run(
    [
        "wget", "-q",
        "https://raw.githubusercontent.com/18217265596/sx/master/extract.py",
        "-O", str(EXTRACT_PY),
    ],
    check=True,
)

if not EXTRACT_PY.exists() or EXTRACT_PY.stat().st_size == 0:
    raise RuntimeError("extract.py 下载失败。")

print("Compatibility patch completed.")
for path in changed:
    print(" ", path)
print("extract.py:", EXTRACT_PY)

In [ ]:
# 4. 上传一个正式生产用 PDB
from google.colab import files

uploaded = files.upload()
pdb_items = [
    (name, data)
    for name, data in uploaded.items()
    if name.lower().endswith(".pdb")
]

if len(pdb_items) != 1:
    raise ValueError("请一次只上传一个 .pdb 文件。")

name, data = pdb_items[0]
INPUT_DIR = ROOT / "user_inputs"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
USER_PDB = INPUT_DIR / Path(name).name
USER_PDB.write_bytes(data)

print("PDB:", USER_PDB)

## 用户参数

下一个代码块是唯一需要用户修改的参数区。所有下载到的 `.pt` 均已编号。`TASK_CHECKPOINT_ID` 决定主任务模型和 checkpoint；编号 15 是侧链打包权重，只通过 `SIDECHAIN_CHECKPOINT_ID` 使用，不能作为主任务权重。

In [ ]:
# 5. 所有用户可指定参数（只需修改本代码块）

# ---------- 权重编号表：全部 15 个 .pt ----------
CHECKPOINT_OPTIONS = {
    1:  ("protein_mpnn", "proteinmpnn_v_48_002.pt", "ProteinMPNN, 0.02 Å noise"),
    2:  ("protein_mpnn", "proteinmpnn_v_48_010.pt", "ProteinMPNN, 0.10 Å noise"),
    3:  ("protein_mpnn", "proteinmpnn_v_48_020.pt", "ProteinMPNN, 0.20 Å noise"),
    4:  ("protein_mpnn", "proteinmpnn_v_48_030.pt", "ProteinMPNN, 0.30 Å noise"),
    5:  ("ligand_mpnn", "ligandmpnn_v_32_005_25.pt", "LigandMPNN, 0.05 Å noise, 25 ligand atoms"),
    6:  ("ligand_mpnn", "ligandmpnn_v_32_010_25.pt", "LigandMPNN, 0.10 Å noise, 25 ligand atoms"),
    7:  ("ligand_mpnn", "ligandmpnn_v_32_020_25.pt", "LigandMPNN, 0.20 Å noise, 25 ligand atoms"),
    8:  ("ligand_mpnn", "ligandmpnn_v_32_030_25.pt", "LigandMPNN, 0.30 Å noise, 25 ligand atoms"),
    9:  ("per_residue_label_membrane_mpnn", "per_residue_label_membrane_mpnn_v_48_020.pt", "MembraneMPNN, per-residue labels"),
    10: ("global_label_membrane_mpnn", "global_label_membrane_mpnn_v_48_020.pt", "MembraneMPNN, global label"),
    11: ("soluble_mpnn", "solublempnn_v_48_002.pt", "SolubleMPNN, 0.02 Å noise"),
    12: ("soluble_mpnn", "solublempnn_v_48_010.pt", "SolubleMPNN, 0.10 Å noise"),
    13: ("soluble_mpnn", "solublempnn_v_48_020.pt", "SolubleMPNN, 0.20 Å noise"),
    14: ("soluble_mpnn", "solublempnn_v_48_030.pt", "SolubleMPNN, 0.30 Å noise"),
    15: ("sidechain_packer", "ligandmpnn_sc_v_32_002_16.pt", "侧链打包权重；不能作为主任务"),
}

print("可用权重编号：")
for number, (model, filename, description) in CHECKPOINT_OPTIONS.items():
    print(f"{number:>2}: {filename:<48} | {description}")

# ---------- 主任务选择 ----------
TASK_CHECKPOINT_ID = 6       # 1–14；普通 binder 常用 2/3，含配体体系常用 6
CHAINS_TO_DESIGN = "A"       # 多链写 "A,B"；空字符串表示设计所有蛋白链

# ---------- LigandMPNN 生成参数 ----------
SEED = 112
TEMPERATURE = 0.10
BATCH_SIZE = 10
NUMBER_OF_BATCHES = 10       # 总序列数 = BATCH_SIZE × NUMBER_OF_BATCHES
PARSE_ATOMS_WITH_ZERO_OCCUPANCY = 1
SAVE_STATS = 1
FIXED_RESIDUES = ""          # 例如 "A12 A13 A25"
REDESIGNED_RESIDUES = ""     # 例如 "A12 A13 A25"；不要与 FIXED_RESIDUES 同时使用
VERBOSE = 1

# ---------- 可选侧链打包 ----------
PACK_SIDE_CHAINS = False
SIDECHAIN_CHECKPOINT_ID = 15
NUMBER_OF_PACKS_PER_DESIGN = 1

# ---------- extract.py 参数 ----------
EXTRACT_SOURCE_GLOB = "*.fa"                  # 从 USER_OUT/seqs 中合并哪些 FASTA
EXTRACT_TOP_N = 20                            # 对应 extract.py 的 --top/-n
EXTRACT_COMBINED_FASTA_NAME = "all_sequences.fa"
EXTRACT_TSV_NAME = "top_unique_sequences.tsv"
EXTRACT_FASTA_NAME = "top_unique_sequences.fa"

# ---------- 输出 ----------
DOWNLOAD_RESULTS_ZIP = True
RESULT_ZIP_NAME = "ligandmpnn_results"

# ---------- 参数解析与检查：以下一般无需修改 ----------
if TASK_CHECKPOINT_ID not in range(1, 15):
    raise ValueError("TASK_CHECKPOINT_ID 必须为 1–14；15 仅用于侧链打包。")

MODEL_TYPE, CHECKPOINT_NAME, TASK_DESCRIPTION = CHECKPOINT_OPTIONS[TASK_CHECKPOINT_ID]
CHECKPOINT_PATH = MODEL_DIR / CHECKPOINT_NAME

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"未找到主任务权重：{CHECKPOINT_PATH}")

if FIXED_RESIDUES.strip() and REDESIGNED_RESIDUES.strip():
    raise ValueError("FIXED_RESIDUES 与 REDESIGNED_RESIDUES 不能同时使用。")

if EXTRACT_TOP_N < 1:
    raise ValueError("EXTRACT_TOP_N 必须至少为 1。")

if PACK_SIDE_CHAINS:
    if SIDECHAIN_CHECKPOINT_ID != 15:
        raise ValueError("当前侧链打包权重编号必须为 15。")
    _, SIDECHAIN_CHECKPOINT_NAME, _ = CHECKPOINT_OPTIONS[SIDECHAIN_CHECKPOINT_ID]
    SIDECHAIN_CHECKPOINT_PATH = MODEL_DIR / SIDECHAIN_CHECKPOINT_NAME
    if not SIDECHAIN_CHECKPOINT_PATH.exists():
        raise FileNotFoundError(f"未找到侧链打包权重：{SIDECHAIN_CHECKPOINT_PATH}")
else:
    SIDECHAIN_CHECKPOINT_PATH = None

print("\n当前任务：")
print(" Model type:", MODEL_TYPE)
print(" Checkpoint:", CHECKPOINT_PATH)
print(" Description:", TASK_DESCRIPTION)
print(" Chains to design:", CHAINS_TO_DESIGN or "all")
print(" Total sequences:", BATCH_SIZE * NUMBER_OF_BATCHES)
print(" Pack side chains:", PACK_SIDE_CHAINS)
print(" extract.py input glob:", EXTRACT_SOURCE_GLOB)
print(" extract.py top N:", EXTRACT_TOP_N)

In [ ]:
# 6. 定义完整日志运行函数
from typing import Sequence

def run_and_show(command: Sequence[str], cwd: Path = ROOT):
    env = os.environ.copy()
    env["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
    env["PYTHONUNBUFFERED"] = "1"

    print("Running command:\n")
    print(" ".join(map(str, command)))
    print("\n" + "=" * 90)

    result = subprocess.run(
        list(map(str, command)),
        cwd=str(cwd),
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    print(result.stdout)
    print("=" * 90)
    print("Exit status:", result.returncode)

    if result.returncode != 0:
        raise RuntimeError("命令运行失败，完整报错见上方。")

    return result

In [ ]:
# 7. 运行正式生产任务
USER_OUT = ROOT / "outputs" / USER_PDB.stem
shutil.rmtree(USER_OUT, ignore_errors=True)

checkpoint_flags = {
    "protein_mpnn": "--checkpoint_protein_mpnn",
    "ligand_mpnn": "--checkpoint_ligand_mpnn",
    "soluble_mpnn": "--checkpoint_soluble_mpnn",
    "per_residue_label_membrane_mpnn": "--checkpoint_per_residue_label_membrane_mpnn",
    "global_label_membrane_mpnn": "--checkpoint_global_label_membrane_mpnn",
}

command = [
    sys.executable, "-u", "run.py",
    "--model_type", MODEL_TYPE,
    checkpoint_flags[MODEL_TYPE], str(CHECKPOINT_PATH),
    "--seed", str(SEED),
    "--pdb_path", str(USER_PDB),
    "--out_folder", str(USER_OUT),
    "--batch_size", str(BATCH_SIZE),
    "--number_of_batches", str(NUMBER_OF_BATCHES),
    "--temperature", str(TEMPERATURE),
    "--parse_atoms_with_zero_occupancy", str(PARSE_ATOMS_WITH_ZERO_OCCUPANCY),
    "--save_stats", str(SAVE_STATS),
    "--verbose", str(VERBOSE),
]

if CHAINS_TO_DESIGN.strip():
    command.extend(["--chains_to_design", CHAINS_TO_DESIGN.strip()])

if FIXED_RESIDUES.strip():
    command.extend(["--fixed_residues", FIXED_RESIDUES.strip()])

if REDESIGNED_RESIDUES.strip():
    command.extend(["--redesigned_residues", REDESIGNED_RESIDUES.strip()])

if PACK_SIDE_CHAINS:
    command.extend([
        "--pack_side_chains", "1",
        "--checkpoint_path_sc", str(SIDECHAIN_CHECKPOINT_PATH),
        "--number_of_packs_per_design", str(NUMBER_OF_PACKS_PER_DESIGN),
    ])

run_and_show(command)
print("Production output:", USER_OUT)

In [ ]:
# 8. 调用 extract.py 提取 overall_confidence 最高的唯一序列
SEQ_DIR = USER_OUT / "seqs"
source_fastas = sorted(SEQ_DIR.glob(EXTRACT_SOURCE_GLOB))

if not source_fastas:
    raise FileNotFoundError(
        f"在 {SEQ_DIR} 中没有匹配 {EXTRACT_SOURCE_GLOB!r} 的 FASTA。"
    )

COMBINED_FASTA = USER_OUT / EXTRACT_COMBINED_FASTA_NAME
with COMBINED_FASTA.open("w", encoding="utf-8") as output:
    for fasta in source_fastas:
        text = fasta.read_text(encoding="utf-8")
        output.write(text)
        if text and not text.endswith("\n"):
            output.write("\n")

EXTRACT_TSV = USER_OUT / EXTRACT_TSV_NAME
extract_command = [
    sys.executable, str(EXTRACT_PY),
    "--input", str(COMBINED_FASTA),
    "--top", str(EXTRACT_TOP_N),
]

extract_result = subprocess.run(
    extract_command,
    cwd=str(ROOT),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

if extract_result.returncode != 0:
    print(extract_result.stderr)
    raise RuntimeError("extract.py 运行失败。")

EXTRACT_TSV.write_text(extract_result.stdout, encoding="utf-8")

EXTRACT_FASTA = USER_OUT / EXTRACT_FASTA_NAME
with EXTRACT_FASTA.open("w", encoding="utf-8") as handle:
    for line in extract_result.stdout.splitlines():
        if not line.strip():
            continue
        rank, confidence, record_id, sequence = line.split("\t", 3)
        handle.write(
            f">rank={rank}, overall_confidence={confidence}, id={record_id}\n"
            f"{sequence}\n"
        )

print("Combined input:", COMBINED_FASTA)
print("extract.py command:", " ".join(extract_command))
print("TSV output:", EXTRACT_TSV)
print("FASTA output:", EXTRACT_FASTA)
print("\nExtracted records:\n")
print(extract_result.stdout)

In [ ]:
# 9. 打包并下载全部结果
from google.colab import files

archive_path = shutil.make_archive(
    f"/content/{RESULT_ZIP_NAME}",
    "zip",
    root_dir=str(USER_OUT),
)

print("Archive:", archive_path)

if DOWNLOAD_RESULTS_ZIP:
    files.download(archive_path)